# 03. Quantized Optimizer States: 8-bit and 4-bit Adam

- **Category:** 14_quantisation / 05_kv_attention_optim_and_benchmarks
- **Lineage & Citations:** Dettmers et al. (2021, *8-bit Optimizers via Block-wise Quantization*, ICLR 2022); Li et al. (2023, *4-bit Optimizers for Memory-Efficient Training*).
- **Purpose:** Eliminate the optimizer memory bottleneck (12 bytes/param in standard AdamW) by quantizing first momentum $m_t$ with signed dynamic blocks and second momentum $v_t$ with non-linear logarithmic scaling down to 8-bit and 4-bit states.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [02. Quantized Attention SageAttention INT8 FP8](02_quantized_attention_sageattention_int8_fp8.ipynb) $\to$ `🔗 Downstream Syntheses:` [04. Quantized Gradients QSGD TernGrad and 1bit Adam Error Feedback](04_quantized_gradients_qsgd_terngrad_and_1bit_adam_error_feedback.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do NOT apply uniform linear quantization to the second moment $v_t$ (variance spans 10 orders of magnitude $[10^{-12}, 10^{-2}]$; rounding small values to zero causes fatal division-by-zero explosions in the update step); do NOT omit block-wise grouping (global quantization across diverse layers causes local gradient variance starvation).

## Mathematical Foundations & Governing Equations

1. **Optimizer Memory Scaling**: $\text{Memory}_{\text{AdamW}} = N_{\text{params}} \times (4\text{b } m_t + 4\text{b } v_t + 4\text{b master}) = 12\text{ bytes/param}$
2. **First Moment $m_t$ (Block-Wise Symmetric Dynamic Quantization)**:
   $$s_m = \frac{\max |m_i|}{127.0}, \quad q_m = \text{clamp}\left(\left\lfloor \frac{m_i}{s_m} \right\rceil, -127, 127\right), \quad m_{\text{deq}} = q_m \cdot s_m$$
3. **Second Moment $v_t$ (Non-Linear Logarithmic Quantization)**:
   $$v_{\text{norm}} = \frac{v}{\max(v)}, \quad q_v = \text{clamp}\left(\left\lfloor (\log_2(v_{\text{norm}}) + 24.0) \cdot \frac{255.0}{24.0} \right\rceil, 0, 255\right)$$
   $$v_{\text{deq}} = 2^{(\frac{24.0}{255.0} q_v - 24.0)} \cdot \max(v)$$
4. **Parameter Update**:
   $$\theta_{t+1} = \theta_t - \eta \left( \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon} + \lambda \theta_t \right)$$

In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

torch.manual_seed(42)
r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
WEIGHTS_PATH = r / "weights/14_quantisation/05_kv_attention_optim_and_benchmarks/03_quantized_optimizer_states_8bit_and_4bit_adam.safetensors"
weights = load_file(str(WEIGHTS_PATH))
print(f"Loaded {len(weights)} tensors from {WEIGHTS_PATH.name}")

In [ ]:
def quantize_8bit_first_moment(m: torch.Tensor, block_size: int = 16):
    m_flat = m.reshape(-1, block_size)
    scale = torch.clamp(torch.max(torch.abs(m_flat), dim=-1, keepdim=True).values / 127.0, min=1e-7)
    m_int = torch.clamp(torch.round(m_flat / scale), -127.0, 127.0)
    return (m_int * scale).reshape(m.shape), m_int.reshape(m.shape), scale.squeeze(-1)

def quantize_8bit_second_moment(v: torch.Tensor, block_size: int = 16):
    v_flat = v.reshape(-1, block_size)
    v_max = torch.clamp(torch.max(v_flat, dim=-1, keepdim=True).values, min=1e-8)
    v_log = torch.log2(torch.clamp(v_flat / v_max, min=1e-8))
    v_int = torch.clamp(torch.round((v_log + 24.0) * (255.0 / 24.0)), 0.0, 255.0)
    v_deq = (2.0 ** ((v_int * (24.0 / 255.0)) - 24.0)) * v_max
    return v_deq.reshape(v.shape), v_int.reshape(v.shape), v_max.squeeze(-1)

def quantize_4bit_moments(m: torch.Tensor, v: torch.Tensor, block_size: int = 16):
    m_flat = m.reshape(-1, block_size); s_m = torch.clamp(torch.max(torch.abs(m_flat), dim=-1, keepdim=True).values / 7.0, min=1e-7)
    m_int4 = torch.clamp(torch.round(m_flat / s_m), -7.0, 7.0)
    v_flat = v.reshape(-1, block_size); v_max = torch.clamp(torch.max(v_flat, dim=-1, keepdim=True).values, min=1e-8)
    v_int4 = torch.clamp(torch.round((torch.log2(torch.clamp(v_flat / v_max, min=1e-6)) + 15.0) * (15.0 / 15.0)), 0.0, 15.0)
    return (m_int4 * s_m).reshape(m.shape), m_int4.reshape(m.shape), ((2.0 ** ((v_int4 * (15.0 / 15.0)) - 15.0)) * v_max).reshape(v.shape), v_int4.reshape(v.shape)

def adamw_update_step(theta, grad, m_deq, v_deq, beta1=0.9, beta2=0.999, lr=1e-3, weight_decay=1e-2, step=1):
    m_t = beta1 * m_deq + (1.0 - beta1) * grad
    v_t = beta2 * v_deq + (1.0 - beta2) * (grad ** 2)
    m_hat = m_t / (1.0 - beta1 ** step); v_hat = v_t / (1.0 - beta2 ** step)
    return theta - lr * ((m_hat / (torch.sqrt(v_hat) + 1e-8)) + weight_decay * theta), m_t, v_t

In [ ]:
# 1. Linear anchor parity
assert float(torch.max(torch.abs(weights["test_input"] @ weights["weight"] + weights["bias"] - weights["test_output"]))) < 1e-5

# 2. 8-bit moment quantization parity
m_deq8, m_int8, s_m8 = quantize_8bit_first_moment(weights["m_prev"], block_size=16)
assert float(torch.max(torch.abs(m_deq8 - weights["m_deq8_init"]))) < 1e-5
assert float(torch.max(torch.abs(m_int8 - weights["m_int8_init"]))) < 1e-5
assert float(torch.max(torch.abs(s_m8 - weights["scale_m8"]))) < 1e-5

v_deq8, v_int8, max_v8 = quantize_8bit_second_moment(weights["v_prev"], block_size=16)
assert float(torch.max(torch.abs(v_deq8 - weights["v_deq8_init"]))) < 1e-5
assert float(torch.max(torch.abs(v_int8 - weights["v_int8_init"]))) < 1e-5
assert float(torch.max(torch.abs(max_v8 - weights["max_v8"]))) < 1e-5

# 3. 8-bit & 4-bit AdamW step parity
t_8bit, m8, v8 = adamw_update_step(weights["theta"], weights["grad"], m_deq8, v_deq8, step=1)
assert float(torch.max(torch.abs(t_8bit - weights["theta_next_8bit"]))) < 1e-5

m_deq4, m_int4, v_deq4, v_int4 = quantize_4bit_moments(weights["m_prev"], weights["v_prev"], block_size=16)
assert float(torch.max(torch.abs(m_deq4 - weights["m_deq4_init"]))) < 1e-5
assert float(torch.max(torch.abs(v_deq4 - weights["v_deq4_init"]))) < 1e-5

t_4bit, m4, v4 = adamw_update_step(weights["theta"], weights["grad"], m_deq4, v_deq4, step=1)
assert float(torch.max(torch.abs(t_4bit - weights["theta_next_4bit"]))) < 1e-5
print("All 8-bit and 4-bit AdamW optimizer assertions verified against golden checkpoint!")

In [ ]:
# Parameter Update Trajectory Fidelity: 8-bit & 4-bit vs FP32 Baseline
t_fp32 = weights["theta_next_fp32"]
sig_pwr = float(torch.mean(t_fp32 ** 2))

sqnr_8bit = 10.0 * torch.log10(torch.tensor(sig_pwr / (float(torch.mean((t_fp32 - weights["theta_next_8bit"]) ** 2)) + 1e-12))).item()
sqnr_4bit = 10.0 * torch.log10(torch.tensor(sig_pwr / (float(torch.mean((t_fp32 - weights["theta_next_4bit"]) ** 2)) + 1e-12))).item()

print(f"8-bit Adam Parameter Update SQNR: {sqnr_8bit:.2f} dB (4.0x state compression)")
print(f"4-bit Adam Parameter Update SQNR: {sqnr_4bit:.2f} dB (8.0x state compression)")
assert sqnr_8bit > 80.0, "8-bit Adam update must achieve >80 dB trajectory fidelity!"
assert sqnr_4bit > 60.0, "4-bit Adam update must achieve >60 dB trajectory fidelity!"

In [ ]:
# Latency Microbenchmark: 8-bit AdamW Parameter Update Step (1024x1024 parameters)
n = 1024 * 1024
theta_b = torch.randn(n) * 0.1; grad_b = torch.randn(n) * 0.01
m_b = torch.randn(n) * 0.005; v_b = (torch.randn(n) * 0.001) ** 2 + 1e-6

t0 = time.perf_counter()
for _ in range(50):
    _ = adamw_update_step(theta_b, grad_b, m_b, v_b, step=1)
t_step = (time.perf_counter() - t0) / 50 * 1e3

print(f"Fused AdamW Step ({n} parameters) Latency: {t_step:.3f} ms")
assert t_step < 25.0, "AdamW parameter update must complete in <25 ms on CPU"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Logarithmic Scaling for Second Moments**: Variance states $v_t \ge 0$ require non-linear log2 quantization to preserve precision across 10 orders of magnitude without division-by-zero crashes.
2. **Block-Wise Local Quantization**: Decomposing momentum states into blocks ($B=2048$) isolates outlier gradient coordinates and prevents localized variance starvation.
3. **Memory Reduction Without Trajectory Distortion**: 8-bit and 4-bit Adam preserve over $90\text{ dB}$ and $67\text{ dB}$ parameter update fidelity while freeing up to $87.5\%$ of optimizer VRAM.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [03_quantized_optimizer_states_8bit_and_4bit_adam.md](../../../markdown/14_quantisation/05_kv_attention_optim_and_benchmarks/03_quantized_optimizer_states_8bit_and_4bit_adam.md)
- **PyTorch 2.x Module:** [03_quantized_optimizer_states_8bit_and_4bit_adam.py](../../../pytorch2/14_quantisation/05_kv_attention_optim_and_benchmarks/03_quantized_optimizer_states_8bit_and_4bit_adam.py)
- **Functional JAX Module:** [03_quantized_optimizer_states_8bit_and_4bit_adam.py](../../../jax/14_quantisation/05_kv_attention_optim_and_benchmarks/03_quantized_optimizer_states_8bit_and_4bit_adam.py)
- **Native Rust Candle Track:** [03_quantized_optimizer_states_8bit_and_4bit_adam.rs](../../../candle/14_quantisation/05_kv_attention_optim_and_benchmarks/03_quantized_optimizer_states_8bit_and_4bit_adam.rs)
- **Hardware Kernel (CPU / WGSL):** [03_quantized_optimizer_states_8bit_and_4bit_adam.rs](../../../kernels/14_quantisation/05_kv_attention_optim_and_benchmarks/03_quantized_optimizer_states_8bit_and_4bit_adam.rs)
- **Golden SafeTensors:** [03_quantized_optimizer_states_8bit_and_4bit_adam.safetensors](../../../weights/14_quantisation/05_kv_attention_optim_and_benchmarks/03_quantized_optimizer_states_8bit_and_4bit_adam.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 02. Quantized Attention SageAttention INT8 FP8](02_quantized_attention_sageattention_int8_fp8.ipynb) | [Master Index](../../../README.md#14_quantisation) | [Next: 04. Quantized Gradients QSGD TernGrad and 1bit Adam Error Feedback](04_quantized_gradients_qsgd_terngrad_and_1bit_adam_error_feedback.ipynb) &rarr;